# DeepATM reconstruction — full run on a Kaggle GPUReproduces Lee et al., *Cell* 188:5081–5099 (2025) — the full-scale run: all21,715 training rows over the full 3,056-residue sequence, 5 folds, 150 epochswith early stopping. Everything in the repo's committed `outputs/` came from awindowed CPU smoke run and is **not** comparable to the paper (deviation D8).Runbook, including the parts that happen outside this notebook (uploading thesupplement as a private dataset, the accelerator and internet toggles, resumingacross sessions): `docs/kaggle-gpu-run.md` in the repo.**Before running:** sidebar → Accelerator = **GPU T4 x2**, Internet = **On**.Then **Save Version → Save & Run All (Commit)** rather than "Run All" — aninteractive session idles out after 20 minutes and takes `/kaggle/working`with it. Expect 3–5 hours.

## 1. Configuration

In [ ]:
# The repo. Public, so no credentials needed.REPO_URL = "https://github.com/ovationtox-ym/DeepATM-Reconstruction.git"# Table S1 from the paper's supplement, uploaded as a PRIVATE Kaggle dataset# and attached via Add Input. Kaggle slugifies dataset titles, so read this# path off the `ls` in the next cell rather than assuming it.MMC1_INPUT = "/kaggle/input/deepatm-mmc1/mmc1.xlsx"# Continuing a run that hit the 12-hour session cap: attach your previous# version via Add Input -> Notebook Output, then point this at its checkpoints# directory, e.g. "/kaggle/input/deepatm-full-run/checkpoints". None = fresh run.RESUME_FROM = NoneWORK = "/kaggle/working/DeepATM-Reconstruction"

## 2. Environment checkIf `cuda` is False, stop here — `--full-length` refuses to run on CPU, and withgood reason: it is ~100x slower than the windowed path and the run would takedays.

In [ ]:
import subprocess, torchprint(f"torch {torch.__version__}  cuda={torch.cuda.is_available()}")assert torch.cuda.is_available(), (    "No GPU. Sidebar -> Accelerator -> GPU T4 x2, then restart the session.")p = torch.cuda.get_device_properties(0)print(f"  {p.name}, {p.total_memory / 1e9:.0f} GB, capability {p.major}.{p.minor}")print(f"  visible devices: {torch.cuda.device_count()} (only cuda:0 is used)")print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)# Confirm the attached datasets and the real mmc1.xlsx path.print(subprocess.run(["ls", "-R", "/kaggle/input"], capture_output=True, text=True).stdout[:4000])

## 3. Dependencies`gemmi` (mmCIF parsing for the Cα coordinate track) and `pygam` (the M7generalized-additive calibration) are the only two requirements Kaggle's imagelacks.Deliberately **not** `pip install -r requirements.txt`: that pins `torch>=2.4`and would have pip replace the image's CUDA build with whatever it resolves.The verification cell below imports every requirement instead, which catches agenuinely missing package without touching the working one.

In [ ]:
!pip install -q gemmi!pip install -q pygam || pip install -q --no-deps pygam

In [ ]:
import importlibfor mod in ["torch", "numpy", "pandas", "sklearn", "scipy", "openpyxl",            "gemmi", "requests", "certifi", "pygam", "matplotlib", "tqdm",            "yaml", "pytest"]:    try:        m = importlib.import_module(mod)        print(f"  ok   {mod:12s} {getattr(m, '__version__', '')}")    except Exception as exc:        print(f"  FAIL {mod:12s} {exc}")

## 4. Repository and data

In [ ]:
import os, shutil, pathlibif not os.path.isdir(WORK):    !git clone --depth 1 {REPO_URL} {WORK}os.chdir(WORK)!git log --oneline -1# The supplement is Elsevier/Cell Press copyright: .gitignore excludes# data/raw/*, so it arrives from the private dataset rather than the repo.assert os.path.exists(MMC1_INPUT), (    f"{MMC1_INPUT} not found. Check the paths printed by the ls above and fix "    "MMC1_INPUT in cell 1.")pathlib.Path("data/raw").mkdir(parents=True, exist_ok=True)shutil.copy(MMC1_INPUT, "data/raw/mmc1.xlsx")print(f"mmc1.xlsx  {os.path.getsize('data/raw/mmc1.xlsx') / 1e6:.1f} MB")

## 5. Restore checkpoints (only when continuing a run)Skipped when `RESUME_FROM` is None. `run_full.sh` passes `--resume` either way:completed folds are skipped and the in-progress fold restarts from its lastepoch. The resume file carries a fingerprint of the run-defining flags, and`train.py` refuses to resume across a settings change rather than silentlymixing two runs.

In [ ]:
import glob, shutil, pathlibif RESUME_FROM:    pathlib.Path("checkpoints").mkdir(exist_ok=True)    restored = glob.glob(f"{RESUME_FROM}/*.pt")    for src in restored:        shutil.copy(src, "checkpoints/")    print(f"restored {len(restored)} checkpoint files from {RESUME_FROM}")    assert restored, "RESUME_FROM is set but held no .pt files — check the path."else:    print("fresh run")

## 6. Optional: ten-minute smoke runExercises every step of the pipeline — splits, ClinVar, train, ablation,evaluate, RF baseline, eDA — on 800 rows with windowed attention. Worth runninginteractively once to confirm the dataset path and internet access beforespending a 12-hour commit on a typo.Its numbers are **not** comparable to the paper. Leave this cell commented outfor the real run.

In [ ]:
# !EPOCHS=2 SMOKE=1 WORKERS=2 bash scripts/run_full.sh

## 7. The full runSplits → ClinVar ≥2★ subset → train → ablation → evaluate → RF baseline → eDAscores → archive. Logs also land in `outputs/logs/`.`WORKERS=2`, not the default 8: Kaggle gives 4 vCPUs, and 8 loader workersoversubscribe them and slow the run down.

In [ ]:
!WORKERS=2 bash scripts/run_full.sh

## 8. Persist the results`/kaggle/working` is what the committed version saves. The repo was clonedinside it, so `outputs/` and `checkpoints/` are already persisted in place —this cell just lifts the archive and a copy of `outputs/` to the top level sothey are easy to find on the Output tab, and so a resumed session can attach`checkpoints/` directly.

In [ ]:
import glob, shutil, osfor src in glob.glob("deepatm-results-*.tar.gz"):    shutil.copy(src, "/kaggle/working/")    print(f"{src}  {os.path.getsize(src) / 1e6:.1f} MB")shutil.copytree("outputs", "/kaggle/working/outputs", dirs_exist_ok=True)shutil.copytree("checkpoints", "/kaggle/working/checkpoints", dirs_exist_ok=True)print(sorted(os.listdir("/kaggle/working")))

## 9. Headline numbersThe run is comparable to the paper only if the window prints as `full length`and `n_rows` is 21,715. A windowed or subsampled run prints its window sizeinstead — the signal that the result belongs to deviation D8 and not to thereproduction.

In [ ]:
import json, pathlibm = json.loads(pathlib.Path("outputs/metrics.json").read_text())cv, targets = m["cross_validation"], m["paper_targets"]summary = json.loads(pathlib.Path("outputs/train_summary.json").read_text())print(f"  rows trained on   {summary['n_rows']}   paper 21715")w = m.get("ensemble", {}).get("window_size")print(f"  window            {w if w is not None else 'full length (comparable to the paper)'}")print()print(f"  CV Pearson r      {cv['median_pearson']:.3f}   paper {targets['cv_pearson']:.2f}")one = m.get("clinvar_1star", {}).get("deepatm", {})if one.get("auroc"):    print(f"  auROC >=1 star    {one['auroc']:.3f}   paper {targets['test_auroc_1star']:.2f}")two = m.get("clinvar_2star") or {}if two.get("auroc"):    print(f"  auROC >=2 star    {two['auroc']:.3f}   (n={two['n']}, paper n={two.get('paper_n')})")p = pathlib.Path("outputs/predict_summary.json")if p.exists():    eda = json.loads(p.read_text()).get("vs_published_eda", {})    if eda:        print(f"  vs published eDA  {eda['pearson']:.3f}   paper {targets['eda_correlation']:.2f}")a = pathlib.Path("outputs/ablation_comparison.json")if a.exists():    ab = json.loads(a.read_text()).get("paired_bootstrap", {})    if ab:        print(f"  ablation p        {ab.get('p_value')}   paper 0.032")

---Download `deepatm-results-<timestamp>.tar.gz` from the version's **Output** tab.It holds `outputs/` and the five fold checkpoints.If the session hit the 12-hour cap partway through: attach this version via**Add Input → Notebook Output**, set `RESUME_FROM` in cell 1 to its`checkpoints` directory, and commit again. §6 of `docs/kaggle-gpu-run.md`.